# 6. Classification

Predict the 5 NewsGuard levels (logistic regression with balanced classes; ModernBERT with balanced cross-entropy). Same split, same models for both tasks. Linear models: the penalty is chosen with 3 inner folds of the
training outlets. ModernBERT: 5 epochs, lr 3e-5, effective batch 32, last epoch kept. For every test article the
outputs are saved in `predictions/classification/<model>/`, and every fold model in `Modelli/20_FromScratch/`.

In [1]:
TASK = "classification"
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B"     # "ModernBERT" runs on the GPU
N_JOBS = 16

In [2]:
# Parameters
MODELS = "Qwen3-8B,Mistral-24B"


In [3]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import models as MO

art, outlets = MO.load()

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
for name in MODELS.split(","):
    for k in range(1, 6):
        if name == "ModernBERT":
            MO.bert_fold(TASK, art, k, "answerdotai/ModernBERT-base", name, LOG)
        else:
            MO.linear_fold(TASK, name, art, k, X=None if name == "TF-IDF" else MO.load_embeddings(name, art),
                           n_jobs=N_JOBS, logfile=LOG)

2026-11-11 23:17:54  classification, Qwen3-8B, fold 1: penalty 0.01 (inside the grid), 116 s


2026-11-11 23:19:43  classification, Qwen3-8B, fold 2: penalty 0.01 (inside the grid), 109 s


2026-11-11 23:21:28  classification, Qwen3-8B, fold 3: penalty 0.01 (inside the grid), 105 s


2026-11-11 23:22:44  classification, Qwen3-8B, fold 4: penalty 0.0001 (on the grid edge), 75 s


2026-11-11 23:25:15  classification, Qwen3-8B, fold 5: penalty 0.1 (inside the grid), 151 s


2026-11-11 23:27:21  classification, Mistral-24B, fold 1: penalty 0.1 (inside the grid), 126 s


2026-11-11 23:29:23  classification, Mistral-24B, fold 2: penalty 0.1 (inside the grid), 121 s


2026-11-11 23:30:47  classification, Mistral-24B, fold 3: penalty 0.001 (inside the grid), 84 s


2026-11-11 23:31:57  classification, Mistral-24B, fold 4: penalty 0.0001 (on the grid edge), 70 s


2026-11-11 23:33:39  classification, Mistral-24B, fold 5: penalty 0.1 (inside the grid), 101 s


In [5]:
for name in MODELS.split(","):
    folder = C.PREDICTIONS / TASK / name
    if name == "ModernBERT":
        print(pd.concat([pd.read_csv(f).assign(fold=f.stem[-1]) for f in sorted(folder.glob("loss_fold*.csv"))])
              .pivot(index="epoch", columns="fold", values="loss").round(4))
    else:
        p = pd.concat([pd.read_csv(f) for f in sorted(folder.glob("penalty_fold*.csv"))])
        print(name, "chosen penalty per fold:", p[p.chosen].set_index("fold").penalty.to_dict())

Qwen3-8B chosen penalty per fold: {1: 0.01, 2: 0.01, 3: 0.01, 4: 0.0001, 5: 0.1}
Mistral-24B chosen penalty per fold: {1: 0.1, 2: 0.1, 3: 0.001, 4: 0.0001, 5: 0.1}
